# 0-레이어 트랜스포머: 바이그램 통계 - 0-레이어 트랜스포머 구현

- Tutorial ID: `tut-1-1`
- Tutorial: 0-레이어 트랜스포머: 바이그램 통계
- Section ID: `s1-1-1`
- Section: 0-레이어 트랜스포머 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 0-레이어 트랜스포머 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

In [ ]:
# 0-레이어 트랜스포머: 바이그램 모델

In [ ]:
# 간단한 어휘 사전
vocab = ['<BOS>', '안녕', '하세', '요', '반갑', '습니다', '<EOS>', '저는', 'AI', '입니다']
vocab_size = len(vocab)
d_model = 6  # 임베딩 차원

print("=" * 55)
print("0-레이어 트랜스포머: 바이그램 통계 분석")
print("=" * 55)
print(f"어휘 크기: {vocab_size}, 임베딩 차원: {d_model}")
print(f"어휘: {vocab}")

np.random.seed(42)

# 임베딩/언임베딩 행렬 (훈련된 것처럼 초기화)
W_E = np.random.randn(vocab_size, d_model) * 0.5  # (vocab, d_model)
W_U = np.random.randn(d_model, vocab_size) * 0.5  # (d_model, vocab)

In [ ]:
# 핵심: W_U @ W_E = 바이그램 테이블

In [ ]:
# 이 행렬은 모든 (현재 토큰, 다음 토큰) 쌍의 로짓을 담고 있습니다
bigram_table = W_E @ W_U  # (vocab, vocab)

print(f"
바이그램 테이블 shape: {bigram_table.shape}")
print("(행 = 현재 토큰, 열 = 다음 토큰 로짓)")

# 소프트맥스로 확률 변환
def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=axis, keepdims=True)

bigram_probs = softmax(bigram_table)

# 특정 토큰 이후 가장 가능성 높은 다음 토큰 출력
print("
바이그램 확률 (상위 3개):")
for i, word in enumerate(vocab):
        top_k = np.argsort(bigram_probs[i])[-3:][::-1]
    preds = [(vocab[j], f"{bigram_probs[i][j]:.3f}") for j in top_k]
    print(f"  '{word}' 다음: {preds}")

In [ ]:
# 0-레이어 트랜스포머 전방 패스

In [ ]:
def zero_layer_transformer(token_id, W_E, W_U):
    """
    입력: 토큰 ID
    출력: 다음 토큰에 대한 로짓
    
        수식: logits = W_U^T @ W_E[token_id]
    """
    # 임베딩 조회 (원-핫 × W_E = W_E의 행 선택)
    embedding = W_E[token_id]  # (d_model,)
    
    # 언임베딩 (프로젝션)
        logits = embedding @ W_U  # (vocab_size,)
    
    return logits

# 테스트
token_id = vocab.index('안녕')
logits = zero_layer_transformer(token_id, W_E, W_U)
probs = softmax(logits)

print(f"
'안녕' 다음 토큰 예측:")
for i, (word, prob) in enumerate(zip(vocab, probs)):
    bar = '█' * int(prob * 30)
    print(f"  {word:10s}: {bar} {prob:.4f}")

In [ ]:
# 가상 가중치 분석

In [ ]:
print("
가상 가중치 (Virtual Weights) 분석:")
print("W_U @ W_E 의 특이값 분포:")
U, S, Vt = np.linalg.svd(bigram_table)
print(f"  특이값 (상위 5개): {np.round(S[:5], 3)}")
print(f"  실효 랭크: {np.sum(S > 0.1)}")
print(f"  이 행렬의 랭크는 d_model({d_model})로 제한됩니다!")
print(f"  (W_E는 {vocab_size}×{d_model}, W_U는 {d_model}×{vocab_size})")
print(f"  따라서 최대 랭크 = min({d_model}, {vocab_size}) = {d_model}")